In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display
from config.config import DATA_INTERIM, OUTPUTS

# Results and Discussion

This notebook presents the full results of my replication of Suss & Oliveira (2023), *Economic Inequality and the Spatial Distribution of Stop and Search: Evidence from London*, using 2025 data. All figures and tables were produced by pipelines 07 and 08 and are loaded here for presentation and interpretation.

The original paper used London stop and search data from 2019, housing value inequality derived from Zoopla property estimates, and control variables from the 2011 Census and 2019 Index of Multiple Deprivation. My replication uses 2025 Metropolitan and City of London Police S&S data and Land Registry transactions from 2022 to 2024 to construct the Gini coefficient. Control variables are drawn from the 2021 Census and 2025 IMD where available. These methodological deviations are documented throughout and discussed in the summary section.

 The central hypothesis - that economic inequality, measured by housing value dispersion within LSOAs, is positively associated with S&S incidence even after controlling for prior crime rates and neighbourhood characteristics - is the principal finding I seek to replicate.

In [ ]:
# Load all saved outputs
analytical = pd.read_csv(DATA_INTERIM / "analytical_dataset.csv")
table2 = pd.read_csv(OUTPUTS / "tables" / "table2_ols_sdm.csv")
table3 = pd.read_csv(OUTPUTS / "tables" / "table3_negative_binomial.csv")
me_df = pd.read_csv(OUTPUTS / "tables" / "marginal_effects_model2.csv")
table_a2 = pd.read_csv(OUTPUTS / "tables" / "table_a2_rate_robustness.csv")
table_a3 = pd.read_csv(OUTPUTS / "tables" / "table_a3_no_westminster.csv")
s60_pace = pd.read_csv(OUTPUTS / "tables" / "table_s60_pace_robustness.csv")

print("All outputs loaded")

## 1. Descriptive Statistics

Table 1 below presents descriptive statistics for all variables in my analytical dataset, alongside the equivalent figures from Suss & Oliveira (2023) Table 1 for direct comparison. My full dataset contains 4,994 London LSOAs using 2021 boundaries. The regression sample reduces to 3,592 LSOAs after dropping cases with missing values, primarily due to the Gini coefficient threshold of 30 Land Registry transactions per LSOA.

Several differences from the paper's descriptive statistics are worth noting. The mean S&S count in my data is 28.3 per LSOA compared to 47.6 in the paper. This reflects the different time periods: the paper uses the full calendar year 2019, while my data covers the full calendar year of 2025. The maximum of 2,106 is virtually identical to the paper's 2,108, suggesting the same concentration of activity in a small number of high-activity LSOAs persists.

My mean Gini coefficient (0.221) is slightly higher than the paper's (0.198), and my range is wider (0.021-0.719 vs 0.042-0.649). This likely reflects differences between Land Registry transaction data and Zoopla estimated values: Land Registry captures actual sale prices which may show more extreme variation, particularly in areas with infrequent but high-value transactions.

The mean property value (£666k vs £605k) reflects London house price inflation between 2019 and 2022–2024. The mean percentage non-white residents (45.2% vs 39.4%) reflects demographic change captured by the 2021 Census compared to the 2011 Census used in the paper. The drug offence rate is not directly comparable due to the different denominator: I use 2021 workplace population rather than the 2011 figures used in the paper. This produces a substantially lower rate (0.010 vs 0.258), as 2021 workplace populations are larger.

The mean TfL distance is considerably higher in my data (723m vs 404m). This is a methodological difference: I calculate mean distance from actual S&S stop locations to the nearest station, whereas the paper uses LSOA centroids. Stop locations are not uniformly distributed within LSOAs - they tend to cluster near transport hubs and high streets - so centroid-based distances will systematically underestimate proximity to stations.

In [ ]:
desc = analytical[[
    "ss_count", "gini_housing", "mean_price", "pop_density",
    "income_score", "imd_crime_score", "drug_rate_2024",
    "mean_dist_to_tfl_m", "pct_non_white"
]].describe().round(3)

# Paper's Table 1 values for comparison
paper_stats = pd.DataFrame({
    "Variable": [
        "Stops", "Gini coefficient", "Average property value (£)",
        "Density (workday pop/ha)", "Income deprivation score",
        "Crime deprivation score", "Drugs rate",
        "Distance to nearest TfL station (m)", "Non-white (%)"
    ],
    "Our N": [4994, 3704, 4992, 4994, 4994, 4994, 4994, 4866, 4994],
    "Our mean": [28.3, 0.221, 666119, 152.6, 0.271, 0.092, 0.010, 723.4, 45.2],
    "Our SD": [66.8, 0.094, 495868, 108.3, 0.148, 0.632, 0.011, 577.2, 18.7],
    "Our min": [0, 0.021, 150694, 2.6, 0.010, -2.294, 0.000, 8.3, 3.3],
    "Our max": [2106, 0.719, 7789304, 1813.9, 0.998, 2.199, 0.255, 7673.8, 98.0],
    "Paper N": [4835, 4835, 4835, 4835, 4835, 4835, 4831, 4835, 4835],
    "Paper mean": [47.6, 0.198, 604579, 137.0, 0.136, 0.258, 0.258, 404.5, 39.4],
    "Paper SD": [102.3, 0.086, 409549, 117.8, 0.076, 0.571, 0.303, 445.1, 20.4],
})

print("Table 1: Descriptive Statistics — Our Data vs Suss & Oliveira (2023)")
print("=" * 100)
display(paper_stats)
paper_stats.to_csv(OUTPUTS / "tables" / "table1_descriptive_statistics.csv",
                   index=False)
print("Saved table1_descriptive_statistics.csv")

## 2. Spatial Distribution of Stop and Search and Housing Inequality

Figures 1 and 2 below replicate the paper's opening choropleth maps, showing the spatial distribution of S&S activity and housing value inequality across London's 4,994 LSOAs.

**Figure 1** shows a highly heterogeneous distribution of S&S activity. The highest concentrations (200+ stops) are visible in central London, parts of east London and isolated LSOAs in outer boroughs. The majority of LSOAs fall in the 1-25 range, with 128 LSOAs recording zero stops entirely. This pattern broadly mirrors the paper's Figure 1, with the same central and north-east concentration, though my data shows somewhat lower overall intensity consistent with the lower mean count.

**Figure 2** shows housing value inequality concentrated most heavily in inner London — particularly central and south-west London — where expensive and cheaper properties coexist within small geographic units. Grey areas (1,290 LSOAs, 25.8% of the total) indicate LSOAs with fewer than 30 Land Registry transactions in the 2022–2024 period and are excluded from the Gini calculation. This is a higher proportion of missing Gini values than in the paper, which used Zoopla estimated values covering approximately 23 million addresses and achieved near-complete coverage. The grey areas are disproportionately concentrated in outer London, where transaction volumes are lower relative to the housing stock.

The visual comparison of the two maps suggests a spatial association between inequality and S&S activity in central London, consistent with the paper's hypothesis. The formal test of this association follows in the regression models.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

img1 = mpimg.imread(OUTPUTS / "figures" / "fig1_ss_count.png")
img2 = mpimg.imread(OUTPUTS / "figures" / "fig2_gini.png")

axes[0].imshow(img1)
axes[0].set_axis_off()
axes[1].imshow(img2)
axes[1].set_axis_off()

plt.tight_layout()
plt.show()

## 3. Spatial Autocorrelation and the OLS/SDM Models (Table 2)

Before fitting the negative binomial models, the paper follows a two-step strategy to account for potential spatial dependence between LSOAs. First, an OLS model is fitted treating stops as a continuous variable. Moran's I is then computed on the OLS residuals to test for spatial autocorrelation. If autocorrelation is present, the Spatial Durbin Model (SDM) is fitted. The SDM includes spatial lags of both the outcome variable and all regressors, using a row-standardised Queen contiguity weights matrix. Moran's I is then computed on the SDM residuals to confirm autocorrelation has been eliminated.

My OLS residuals produce a Moran's I of 0.219 (p < 0.0001), confirming significant positive spatial autocorrelation — nearby LSOAs have more similar S&S counts than would be expected by chance. This closely mirrors the paper's finding (Moran's I not reported numerically in the paper, but described as significant at p < 0.01). After fitting the SDM, the Moran's I on residuals falls to −0.005 (p = 0.311 via Monte Carlo simulation with 1,000 permutations), confirming that the SDM successfully accounts for spatial dependence. The paper reports a Monte Carlo p-value of 0.587 on SDM residuals. My slightly lower p-value still comfortably exceeds the conventional 0.05 threshold.

The spatial autoregressive parameter rho (ρ = 0.348) confirms meaningful positive spatial dependence in S&S activity. The level of stops in a given LSOA is positively associated with stops in neighbouring LSOAs, consistent with police resource allocation operating at a broader spatial scale than individual LSOAs. The paper reports ρ = 0.4, so my estimate is somewhat lower but in the same direction.

The Gini coefficient is positive and statistically significant in the SDM (p < 0.05), consistent with the paper's finding that inequality predicts S&S incidence after controlling for spatial dependence. However, unlike the paper, the Gini coefficient is not significant in the OLS model. This may reflect the greater role of spatial spillovers in my data - once neighbouring LSOA characteristics are accounted for, the local inequality signal becomes clearer.

The crime deprivation coefficient changes sign between OLS and SDM (positive to negative), which differs from the paper where it remains positive in both. This is likely attributable to the different scaling of the IMD 2025 crime score compared to the 2019 version, combined with the spatial lag structure absorbing the local crime effect into neighbouring area characteristics.

In [ ]:
print("Table 2: OLS and Spatial Durbin Model Results")
print("=" * 75)
display(table2)

## 4. Negative Binomial Regression Models (Table 3)

The negative binomial model is the paper's primary analytical approach. It is appropriate here because S&S counts are overdispersed — the variance (4,461) substantially exceeds the mean (28.3) — which violates the Poisson assumption of equal mean and variance. Three models are estimated following the paper exactly. Model 1 includes only the Gini coefficient and borough fixed effects. Model 2 adds all control variables. Model 3 adds an interaction term between the Gini coefficient and the income deprivation score.

**Model 1** finds a positive but only marginally significant association between inequality and S&S (coef = 0.053, p < 0.05). In the paper, Model 1 produces a coefficient of 0.170 (p < 0.01). The weaker result in my Model 1 is consistent with the less complete coverage of my Gini measure and the different time period.

**Model 2** produces the key finding: the Gini coefficient is positive and highly significant (coef = 0.143, p < 0.001), with an incidence rate ratio (IRR) of 1.153. This means a one standard deviation increase in housing value inequality is associated with a 15.3% increase in the expected number of stops, holding all other variables constant. The paper reports an IRR of 1.33 (33% increase). My effect is in the same direction and statistically robust, but smaller in magnitude. The reduction in effect size plausibly reflects the different Gini data source (Land Registry transactions vs Zoopla estimates) and the different S&S period.

The control variables behave largely as expected. Crime deprivation (coef = 0.607) and drugs rate (coef = 0.286) are both strongly positive — areas with higher prior crime activity attract more S&S, consistent with a crime deterrence motivation. Distance to TfL stations is negative (coef = −0.111), suggesting S&S concentrates near transport hubs where foot traffic is higher. Population density is positive (coef = 0.055), reflecting greater police presence in denser areas. Percentage non-white residents is positive (coef = 0.225), consistent with the ethnic disproportionality in S&S documented by the paper and wider literature. Income deprivation is negative (coef = −0.163). This suggests that — holding inequality constant — more deprived areas actually have fewer stops, which aligns with the paper's argument that it is the mixing of rich and poor rather than poverty per se that drives S&S activity.

**Model 3** introduces the interaction between Gini and income deprivation. In the paper, this interaction is negative and highly significant (coef = −0.097, p < 0.001), indicating that the effect of inequality on S&S is stronger in more affluent areas. In my replication, the interaction coefficient is negative (coef = −0.023) but does not reach statistical significance (p > 0.05). This is the most notable divergence from the paper. It suggests that in 2025 data, the moderating role of affluence in the inequality-S&S relationship is weaker or absent. This could reflect genuine change in policing patterns since 2019, differences in the Gini measure, or the updated IMD 2025 income scores having a different distributional relationship with the Gini compared to 2019. This finding warrants cautious interpretation and would benefit from further investigation with alternative inequality measures.

In [ ]:
print("Table 3: Negative Binomial Regression Results")
print("=" * 100)
display(table3)

## 5. Figure 3: Marginal Effects

Figure 3 replicates the paper's Figure 3, showing the marginal effects at the mean (filled circle) and average partial effect (triangle) for each covariate from Model 2, with 95% bootstrap confidence intervals on the average partial effect.

The marginal effect of the Gini coefficient is positive and the confidence interval excludes zero, confirming the result is statistically meaningful in substantive as well as statistical terms. In absolute terms, a one standard deviation increase in inequality at the mean of all other covariates is associated with approximately 1-3 additional stops per LSOA. The paper reports a range of roughly 7-11 at the mean (MEM approach), reflecting the larger coefficient in the 2019 data.

The drugs rate and crime deprivation show the largest marginal effects, consistent with police resource allocation responding strongly to prior crime activity. The TfL distance effect is modest but negative and property value is close to zero, suggesting affluence per se has little independent effect once inequality is controlled for.

In [ ]:
img3 = mpimg.imread(OUTPUTS / "figures" / "fig3_marginal_effects.png")
fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(img3)
ax.set_axis_off()
plt.tight_layout()
plt.show()

print("\nMarginal Effects — Model 2:")
display(me_df.round(3))

## 6. Figure 4: Interaction Effect

Figure 4 replicates the paper's Figure 4, plotting the predicted number of stops as a function of the Gini coefficient at five levels of income deprivation (-2 to +2 standard deviations), comparing Model 2 (no interaction) and Model 3 (with interaction term). All other covariates are fixed at their means (zero after standardisation).

In the paper's Figure 4, the interaction panel shows clearly diverging predicted stop counts across deprivation levels - relatively affluent areas (low income deprivation) show a much steeper positive relationship between inequality and stops, while deprived areas show a flatter relationship. This visual pattern is the paper's most striking finding: the inequality effect is amplified in wealthier areas.

In my replication, the interaction panel shows much less divergence across deprivation levels, consistent with the non-significant interaction coefficient in Model 3. The predicted stop counts at different deprivation levels remain close together across the Gini range, suggesting a more homogeneous inequality effect that does not vary meaningfully with neighbourhood affluence. The linear panel (Model 2) shows the consistent positive association between inequality and stops across all deprivation levels.

In [ ]:
img4 = mpimg.imread(OUTPUTS / "figures" / "fig4_interaction.png")
fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(img4)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 7. Robustness Checks

### 7.1 Stops Rate per 1,000 Residents (Table A.2)

The first robustness check replicates Table A.2 from the paper, using the log of stops per 1,000 residents as the outcome variable rather than the raw count. This tests whether the results are sensitive to the choice of outcome measure and controls for variation in LSOA population size.

The Gini coefficient remains positive and significant in both the OLS and SDM models using the rate outcome, consistent with the paper's finding that results are robust to this alternative specification. The pattern of coefficients is broadly similar to Table 2, though the magnitude differs as expected given the different outcome scale.

In [ ]:
print("Table A.2: Robustness Check — Stops Rate per 1,000 Residents")
print("=" * 75)
display(table_a2)

### 7.2 Westminster Excluded (Table A.3)

The second robustness check replicates Table A.3, removing Westminster Borough from the sample. Westminster contains some of London's most unequal LSOAs - Mayfair and St James's sit alongside some of the most deprived areas in England - as well as some of the highest S&S counts, driven by the high footfall of central London. The concern is that Westminster's extreme values could be driving the overall inequality result.

The Gini coefficient in Model 2 without Westminster (coef = 0.124, p < 0.001) is only marginally smaller than the full sample result (0.143) and remains highly significant. This closely mirrors the paper's robustness check (0.272 dropping to 0.253). The result confirms that the positive association between inequality and S&S is not an artefact of Westminster's outlier characteristics, but a London-wide pattern.

The interaction term remains non-significant without Westminster, consistent with the full sample Model 3.

In [ ]:
print("Table A.3: Robustness Check — Westminster Excluded")
print("=" * 100)
display(table_a3)

### 7.3 S60 vs PACE Searches (Extension Beyond Original Paper)

This analysis goes beyond the original paper to address one of its stated limitations: the inability to distinguish between searches conducted under PACE (requiring reasonable suspicion) and those conducted under Section 60 of the Criminal Justice and Public Order Act 1994 (suspicion-less). The paper acknowledged this is a limitation, noting that PACE and S60 searches may have different spatial distributions.

My 2025 data allows this distinction. Model 2 is re-estimated separately for PACE and S60 search counts.

The Gini coefficient for PACE searches (coef = 0.144, IRR = 1.154) is virtually identical to the full sample results, indicating that the inequality-S&S relationship is driven entirely by searches conducted under reasonable suspicion. The Gini coefficient for S60 searches is negative and close to zero (coef = -0.027, IRR = 0.974), with no meaningful association between local inequality and the spatial concentration of suspicion-less searches.

The absence of an inequality effect for S60 searches should not be read as a reassuring finding. S60 authorisations remove the reasonable suspicion requirement entirely, and a substantial body of research has documented that suspicion-less searches produce the most pronounced ethnic disproportionality of any form of S&S (Shiner et al., 2018; Bowling and Phillips, 2007). Where PACE searches appear to respond to neighbourhood economic characteristics, S60 searches are more plausibly driven by the racial and ethnic composition of the areas in which authorisations are granted. This represents a form of geographically targeted over-policing of minority communities that operates through command decisions rather than individual officer discretion. The non-white percentage coefficient in the full sample model (coef = 0.225) is consistent with this, though the present analysis cannot separate the S60 and PACE contributions to that coefficient.

What the S60/PACE split does reveal is that the two forms of search reflect different social control mechanisms operating at different levels of the police organisation. PACE searches concentrate in economically unequal neighbourhoods, consistent with the paper's hypothesis that individual officers use S&S to manage the visible juxtaposition of economic groups. S60 searches appear to operate through a separate logic — one that prior research suggests is more directly racialised — and warrant separate investigation beyond the scope of this replication.

In [ ]:
print("Table: S60 vs PACE Robustness Check (Model 2 coefficients)")
print("=" * 100)
display(s60_pace)

## 8. Summary and Comparison with Suss & Oliveira (2023)

### Main finding

My replication confirms the paper's central finding: housing value inequality is positively and significantly associated with S&S incidence at the LSOA level in London, even after controlling for prior crime rates, ethnic composition, population density, deprivation, and spatial dependence between neighbouring areas. A one standard deviation increase in the Gini coefficient is associated with a 15.3% increase in expected stops (IRR = 1.153), compared to 33% in the original paper. The direction and significance of the result replicates; the magnitude is smaller, consistent with the different data sources and time period.

### Spatial autocorrelation

The spatial diagnostic results replicate closely. OLS residuals exhibit significant positive spatial autocorrelation (Moran's I = 0.219, p < 0.0001) and the SDM successfully eliminates this (Moran's I = −0.005, p = 0.311). The spatial autoregressive parameter (ρ = 0.348) is somewhat lower than the paper's (0.4) but confirms meaningful spatial dependence in S&S activity.

### Interaction effect

The paper's finding that the inequality effect is moderated by neighbourhood affluence - stronger in wealthier areas - does not replicate. The interaction between Gini and income deprivation is negative as expected but does not reach statistical significance (p > 0.05). This is the most important divergence from the original paper and merits further investigation.

### Robustness

Results are robust to the exclusion of Westminster and to the use of a rate outcome measure, consistent with the paper. The S60/PACE extension reveals that the inequality effect is concentrated in discretionary (PACE) searches. S60 searches show no association with neighbourhood inequality, which — consistent with existing research on ethnic disproportionality in suspicion-less searches — suggests their spatial distribution is driven by different mechanisms, likely including the racial and ethnic composition of targeted areas. This distinction was not testable in the original paper's data and represents a meaningful extension of the analysis, though it raises further questions that go beyond the scope of this replication.

### Key methodological deviations

| Aspect | Original paper | This replication |
|---|---|---|
| S&S data | 2019 | 2025 |
| Inequality measure | Zoopla estimated values (Sep 2019) | Land Registry transactions (2022–2024) |
| Gini threshold | 50 observations | 30 transactions |
| Census | 2011 | 2021 |
| IMD | 2019 | 2025 |
| TfL distance | LSOA centroids | Actual stop locations |
| Drug rate denominator | 2011 workplace population | 2021 workplace population |
| LSOA boundaries | 2011 | 2021 |
| S60/PACE distinction | Not available | Available and analysed |

### Conclusion

Despite these deviations, the core finding replicates: police officers conduct more searches in more economically unequal neighbourhoods, independently of prior crime activity and other neighbourhood characteristics. This is consistent with the interpretation of S&S as a tool of social control - one that responds to the visible juxtaposition of economic groups rather than to crime alone. The S60/PACE extension strengthens this interpretation by locating the inequality effect specifically in discretionary officer behaviour.

In [ ]:
print("Results notebook complete.")
print(f"Tables saved to: {OUTPUTS / 'tables'}")
print(f"Figures saved to: {OUTPUTS / 'figures'}")